<a href="https://colab.research.google.com/github/Zafar488/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/Zafar488/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# ML-09 — Validation and Research Claim Audit

**Lane:** Refresh / Content Opportunity Scoring  
**Primary metric:** Precision@50  
**Purpose:** Human decision-support for content-review prioritisation  
**Development month:** March 2026  
**Feature window:** March 1–15, 2026  
**Outcome window:** March 16–31, 2026  

This notebook is fully standalone. It does not use `%run` to execute the Week-5 notebook.

It reconstructs the Week-5 Logistic Regression from the same warehouse partition, feature set, target proxy, seed, test size, and preprocessing design. It then:

1. reviews two findings from the FlyRank research paper;
2. compares an unsafe random-row split with the retained grouped-client split;
3. audits the final feature set for leakage;
4. examines real grouped-holdout failures;
5. rewrites the strongest model claim using public-safe language.

All conclusions use observed, measured, directional, and decision-support language.

## 0. Setup

Store a Hugging Face **Read** token in Google Colab Secrets using the name `HF_TOKEN`.

The notebook queries only the March 2026 warehouse partition. The June sample is not used for development.

Aggregate receipts are written to `work/outputs/ml09/`.

In [1]:
%pip install -q duckdb huggingface_hub scikit-learn pandas numpy joblib

In [2]:
from __future__ import annotations

import json
import os
import subprocess
from pathlib import Path
from typing import Any

import duckdb
import joblib
import numpy as np
import pandas as pd

from google.colab import userdata
from huggingface_hub import whoami

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.model_selection import (
    GroupShuffleSplit,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)


# ============================================================
# REPRODUCIBILITY
# ============================================================

SEED = 42
TEST_SIZE = 0.25
TOP_K = 50
CLASSIFICATION_THRESHOLD = 0.50

BASELINE_CTR_WEIGHT = 0.70
BASELINE_VISIBILITY_WEIGHT = 0.30

FEATURE_START = "2026-03-01"
FEATURE_END = "2026-03-15"

OUTCOME_START = "2026-03-16"
OUTCOME_END = "2026-03-31"

np.random.seed(SEED)


# ============================================================
# REPOSITORY
# ============================================================

REPO_URL = "https://github.com/Zafar488/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

current_directory = Path.cwd()

if current_directory.name == REPO_DIR:
    REPO_ROOT = current_directory

else:
    REPO_ROOT = Path("/content") / REPO_DIR

    if not REPO_ROOT.exists():
        print("Repository not found. Cloning repository...")

        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                REPO_URL,
                str(REPO_ROOT),
            ],
            check=True,
        )

    os.chdir(REPO_ROOT)


# ============================================================
# OUTPUTS
# ============================================================

OUTPUT_DIR = Path("work/outputs/ml09")
MODEL_DIR = OUTPUT_DIR / "models"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

METRICS_PATH = OUTPUT_DIR / "ml09_metrics.json"
MODEL_PATH = (
    MODEL_DIR
    / "ml09_grouped_logistic_regression.joblib"
)


# ============================================================
# HUGGING FACE AUTHENTICATION
# ============================================================

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN was not found. Add a Hugging Face Read token "
        "to Colab Secrets and enable notebook access."
    )

hf_user = whoami(token=HF_TOKEN)

print(
    "Hugging Face authentication successful:",
    hf_user.get("name", "authenticated user"),
)


# ============================================================
# DUCKDB
# ============================================================

con = duckdb.connect()

safe_token = HF_TOKEN.replace("'", "''")

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{safe_token}'
    )
    """
)

WAREHOUSE_ROOT = "hf://datasets/FlyRank/internship-warehouse"

MARCH_FACT = (
    "read_parquet("
    f"'{WAREHOUSE_ROOT}/"
    "fact_content_daily_performance/"
    "month=2026-03/*.parquet'"
    ")"
)

print("Working directory:", Path.cwd())
print("Output directory:", OUTPUT_DIR)
print("Feature window:", FEATURE_START, "to", FEATURE_END)
print("Outcome window:", OUTCOME_START, "to", OUTCOME_END)

Repository not found. Cloning repository...
Hugging Face authentication successful: zafar4050
Working directory: /content/flyrank-ml-internship
Output directory: work/outputs/ml09
Feature window: 2026-03-01 to 2026-03-15
Outcome window: 2026-03-16 to 2026-03-31


## 1. Two Paper Findings and My Methodology Questions

### Finding 1 — Growing Content Was Longer and Younger

The paper reports that content with rising impressions was, on average, longer and younger than content with falling impressions. Growing pages averaged approximately **3,180 words** and **184 days of age**, while declining pages averaged approximately **2,311 words** and **230 days of age**.

#### Methodology question 1: Where does the label come from?

The trend label compares impressions in the latest 30-day period with the previous 30-day period. I would ask whether pages with very low impression counts were filtered or stabilised before the trend label was assigned. A small absolute change can create a large percentage movement when the starting count is small.

#### Methodology question 2: Does the design support the claim?

This is an observational cohort comparison rather than a causal or predictive validation experiment. The measured difference describes the observed portfolio, but it does not establish that increasing word count or reducing content age will cause growth. Client context, topic, demand, publication timing, and prior visibility may explain part of the association.

**Public-safe interpretation:** Longer and younger pages were associated with stronger recent impression trends in the observed portfolio. The result is directional and may support review prioritisation, but it is not a causal rule.

---

### Finding 2 — Recently Refreshed Mature Content Showed Stronger Measured Performance

The paper reports that the **31–90 day freshness window** had the strongest stable growth-to-decline ratio. It also reports that mature content refreshed within 30 days had higher measured health and impressions than older content that had not been refreshed recently.

#### Methodology question 1: Where does the refresh label come from?

Freshness is defined as days since the last recorded update. I would ask what type of change qualifies as an update. A full rewrite, a metadata edit, and an automated timestamp change may produce the same freshness value while representing very different interventions.

#### Methodology question 2: Does the design support the claim?

The comparison is observational. Pages selected for refresh may already have stronger historical visibility, greater business value, better editorial quality, or more search demand. This creates possible selection bias because refreshed and untouched pages may not be directly comparable.

A stronger design would compare refreshed and unrefreshed pages with similar prior impressions, position, age, topic, and client context. A time-aware before-and-after design could also test whether the measured change occurred after the refresh.

**Public-safe interpretation:** Recent refresh activity was associated with stronger measured performance among mature pages in the observed portfolio. This is a directional decision-support signal rather than proof that refreshing any page will create the same result.

In [3]:
paper_findings = pd.DataFrame(
    [
        {
            "finding": (
                "Growing content was longer and younger"
            ),
            "reported_measure_1": (
                "3,180 vs 2,311 average words"
            ),
            "reported_measure_2": (
                "184 vs 230 average age in days"
            ),
            "label_source": (
                "Latest 30-day impression trend compared "
                "with the previous 30-day period"
            ),
            "evidence_type": (
                "Observational cohort comparison"
            ),
            "methodology_risk": (
                "Low-count instability and confounding"
            ),
            "safe_interpretation": (
                "Longer and younger pages were associated with "
                "stronger recent impression trends in the "
                "observed portfolio."
            ),
        },
        {
            "finding": (
                "Recently refreshed mature content showed "
                "stronger measured performance"
            ),
            "reported_measure_1": (
                "3.2x health comparison"
            ),
            "reported_measure_2": (
                "57x impression comparison"
            ),
            "label_source": (
                "Days since the last recorded content update"
            ),
            "evidence_type": (
                "Observational freshness comparison"
            ),
            "methodology_risk": (
                "Update-definition ambiguity and selection bias"
            ),
            "safe_interpretation": (
                "Recent refresh activity was associated with "
                "stronger measured performance among mature pages."
            ),
        },
    ]
)

display(paper_findings)

assert len(paper_findings) == 2

assert paper_findings[
    "safe_interpretation"
].str.contains(
    "associated",
    case=False,
).all()

assert paper_findings[
    "evidence_type"
].str.contains(
    "observational",
    case=False,
).all()

print(
    "Two paper findings and constructive "
    "methodology questions documented."
)

,finding,reported_measure_1,reported_measure_2,label_source,evidence_type,methodology_risk,safe_interpretation
0,Growing content was longer and younger,"3,180 vs 2,311 average words",184 vs 230 average age in days,Latest 30-day impression trend compared with t...,Observational cohort comparison,Low-count instability and confounding,Longer and younger pages were associated with ...
1,Recently refreshed mature content showed stron...,3.2x health comparison,57x impression comparison,Days since the last recorded content update,Observational freshness comparison,Update-definition ambiguity and selection bias,Recent refresh activity was associated with st...


Two paper findings and constructive methodology questions documented.


## 2. My Model Under an Honest Split — Before and After

The Week-5 Logistic Regression is reconstructed directly in this notebook. This avoids the `%run` notebook-execution error and makes the audit reproducible from a fresh Colab runtime.

### Before — Random Row Split

The random-row split is included as an unsafe comparison design. It can place pages from the same anonymised client in both training and validation. Shared site structure, measurement patterns, and editorial practices may make the result optimistic.

### After — Grouped Client Split

The grouped split places every anonymised client entirely in either training or validation. Client overlap must equal zero.

The retained question is:

> Can the model rank pages for clients that were not observed during training?

Both designs use:

- the same March 2026 operational population;
- the same seven Week-5 features;
- the same target proxy;
- the same Logistic Regression pipeline;
- the same seed and test-size setting;
- Precision@50, Average Precision, and ROC-AUC;
- the positive base rate beside the metrics.

The grouped result is retained as the honest decision-support estimate. The random result is reported only as a validation-design contrast.

In [4]:
# ============================================================
# VERIFY SOURCE PARTITION
# ============================================================

source_check = con.sql(
    f"""
    SELECT
        COUNT(*) AS total_rows,
        MIN(report_date) AS minimum_date,
        MAX(report_date) AS maximum_date,
        COUNT(DISTINCT client_hash_id) AS unique_clients,
        COUNT(
            DISTINCT (
                client_hash_id,
                content_hash_id
            )
        ) AS unique_client_pages
    FROM {MARCH_FACT}
    """
).df()

display(source_check)

minimum_date = pd.to_datetime(
    source_check.loc[0, "minimum_date"]
).date()

maximum_date = pd.to_datetime(
    source_check.loc[0, "maximum_date"]
).date()

assert str(minimum_date) == FEATURE_START
assert str(maximum_date) == OUTCOME_END


# ============================================================
# BUILD ONE ROW PER ANONYMISED CLIENT-PAGE
# ============================================================

page_frame = con.sql(
    f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(
            CASE
                WHEN report_date BETWEEN
                     DATE '{FEATURE_START}'
                     AND DATE '{FEATURE_END}'
                 AND gsc_data_available IS TRUE
                THEN COALESCE(gsc_impressions, 0)
                ELSE 0
            END
        ) AS feature_impressions,

        SUM(
            CASE
                WHEN report_date BETWEEN
                     DATE '{FEATURE_START}'
                     AND DATE '{FEATURE_END}'
                 AND gsc_data_available IS TRUE
                THEN COALESCE(gsc_clicks, 0)
                ELSE 0
            END
        ) AS feature_clicks,

        AVG(
            CASE
                WHEN report_date BETWEEN
                     DATE '{FEATURE_START}'
                     AND DATE '{FEATURE_END}'
                 AND gsc_data_available IS TRUE
                THEN gsc_avg_position
            END
        ) AS feature_avg_position,

        STDDEV_SAMP(
            CASE
                WHEN report_date BETWEEN
                     DATE '{FEATURE_START}'
                     AND DATE '{FEATURE_END}'
                 AND gsc_data_available IS TRUE
                THEN gsc_avg_position
            END
        ) AS feature_position_volatility,

        COUNT(
            DISTINCT CASE
                WHEN report_date BETWEEN
                     DATE '{FEATURE_START}'
                     AND DATE '{FEATURE_END}'
                 AND gsc_data_available IS TRUE
                 AND COALESCE(gsc_impressions, 0) > 0
                THEN report_date
            END
        ) AS feature_active_days,

        COUNT(
            DISTINCT CASE
                WHEN report_date BETWEEN
                     DATE '{FEATURE_START}'
                     AND DATE '{FEATURE_END}'
                 AND gsc_data_available IS TRUE
                THEN report_date
            END
        ) AS feature_available_days,

        SUM(
            CASE
                WHEN report_date BETWEEN
                     DATE '{OUTCOME_START}'
                     AND DATE '{OUTCOME_END}'
                 AND gsc_data_available IS TRUE
                THEN COALESCE(gsc_impressions, 0)
                ELSE 0
            END
        ) AS outcome_impressions,

        COUNT(
            DISTINCT CASE
                WHEN report_date BETWEEN
                     DATE '{OUTCOME_START}'
                     AND DATE '{OUTCOME_END}'
                 AND gsc_data_available IS TRUE
                THEN report_date
            END
        ) AS outcome_available_days

    FROM {MARCH_FACT}

    GROUP BY
        client_hash_id,
        content_hash_id
    """
).df()

duplicate_rows = int(
    page_frame.duplicated(
        subset=[
            "client_hash_id",
            "content_hash_id",
        ]
    ).sum()
)

print("Raw page rows:", f"{len(page_frame):,}")
print("Duplicate client-page rows:", duplicate_rows)

assert duplicate_rows == 0


# ============================================================
# FEATURE ENGINEERING AND AUDIT TARGET
# ============================================================

page_frame["feature_ctr"] = np.where(
    page_frame["feature_impressions"] > 0,
    (
        page_frame["feature_clicks"]
        / page_frame["feature_impressions"]
    ),
    np.nan,
)

page_frame["feature_daily_impressions"] = np.where(
    page_frame["feature_available_days"] > 0,
    (
        page_frame["feature_impressions"]
        / page_frame["feature_available_days"]
    ),
    np.nan,
)

page_frame["outcome_daily_impressions"] = np.where(
    page_frame["outcome_available_days"] > 0,
    (
        page_frame["outcome_impressions"]
        / page_frame["outcome_available_days"]
    ),
    np.nan,
)

page_frame["is_declining_proxy"] = (
    page_frame["outcome_daily_impressions"]
    <
    0.80
    * page_frame["feature_daily_impressions"]
).astype(int)

page_frame["log_feature_impressions"] = np.log1p(
    page_frame["feature_impressions"]
)

page_frame["position_band"] = pd.cut(
    page_frame["feature_avg_position"],
    bins=[
        0,
        3,
        10,
        20,
    ],
    labels=[
        "Top 3",
        "Page 1",
        "Page 2",
    ],
    include_lowest=True,
)

page_frame["position_band"] = (
    page_frame["position_band"]
    .astype("object")
)


# ============================================================
# SAME OPERATIONAL POPULATION AS ML-08
# ============================================================

model_frame = page_frame[
    (page_frame["feature_impressions"] >= 500)
    & (page_frame["feature_available_days"] >= 5)
    & (page_frame["outcome_available_days"] >= 5)
    & (page_frame["feature_avg_position"] > 0)
    & (page_frame["feature_avg_position"] <= 20)
    & (page_frame["feature_ctr"].notna())
    & (page_frame["position_band"].notna())
].copy()

model_frame = (
    model_frame
    .drop_duplicates(
        subset=[
            "client_hash_id",
            "content_hash_id",
        ]
    )
    .reset_index(drop=True)
)

numeric_features = [
    "log_feature_impressions",
    "feature_clicks",
    "feature_ctr",
    "feature_avg_position",
    "feature_active_days",
    "feature_position_volatility",
]

categorical_features = [
    "position_band",
]

feature_columns = (
    numeric_features
    + categorical_features
)

target_column = "is_declining_proxy"
group_column = "client_hash_id"

print(
    "Operational evaluation rows:",
    f"{len(model_frame):,}",
)

print(
    "Observed positive base rate:",
    round(
        model_frame[target_column].mean(),
        3,
    ),
)

print(
    "Anonymised clients:",
    model_frame[group_column].nunique(),
)

assert len(model_frame) > TOP_K
assert target_column not in feature_columns
assert group_column not in feature_columns
assert "content_hash_id" not in feature_columns
assert "outcome_impressions" not in feature_columns
assert "outcome_daily_impressions" not in feature_columns

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,minimum_date,maximum_date,unique_clients,unique_client_pages
0,9841378,2026-03-01,2026-03-31,55,331437


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Raw page rows: 331,437
Duplicate client-page rows: 0
Operational evaluation rows: 34,038
Observed positive base rate: 0.309
Anonymised clients: 32


In [5]:
# ============================================================
# TRAINING-ONLY PREPROCESSING
# ============================================================

numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            ),
        ),
        (
            "scaler",
            StandardScaler(),
        ),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            ),
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numeric_features,
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features,
        ),
    ],
    remainder="drop",
)


def build_audit_model() -> Pipeline:
    return Pipeline(
        steps=[
            (
                "preprocessor",
                clone(preprocessor),
            ),
            (
                "model",
                LogisticRegression(
                    max_iter=2000,
                    class_weight="balanced",
                    random_state=SEED,
                ),
            ),
        ]
    )


def prepare_arrays(
    labels: Any,
    scores: Any,
) -> tuple[np.ndarray, np.ndarray]:
    labels_array = np.asarray(
        labels,
        dtype=int,
    )

    scores_array = np.asarray(
        scores,
        dtype=float,
    )

    if len(labels_array) != len(scores_array):
        raise ValueError(
            "Labels and scores must have equal length."
        )

    if len(labels_array) == 0:
        raise ValueError(
            "Evaluation arrays cannot be empty."
        )

    if not np.isfinite(scores_array).all():
        raise ValueError(
            "Scores contain NaN or infinite values."
        )

    return labels_array, scores_array


def precision_at_k(
    labels: Any,
    scores: Any,
    k: int = TOP_K,
) -> float:
    labels_array, scores_array = prepare_arrays(
        labels,
        scores,
    )

    effective_k = min(
        k,
        len(labels_array),
    )

    ranked_indices = np.argsort(
        -scores_array,
        kind="mergesort",
    )[:effective_k]

    return float(
        labels_array[ranked_indices].mean()
    )


def evaluate_split(
    split_name: str,
    validation_data: pd.DataFrame,
    scores: np.ndarray,
    training_rows: int,
    training_clients: int,
    validation_clients: int,
    client_overlap: int,
) -> dict[str, Any]:
    labels = validation_data[
        target_column
    ].to_numpy(dtype=int)

    return {
        "validation_design": split_name,
        "train_rows": int(training_rows),
        "validation_rows": int(
            len(validation_data)
        ),
        "train_clients": int(
            training_clients
        ),
        "validation_clients": int(
            validation_clients
        ),
        "client_overlap": int(
            client_overlap
        ),
        "positive_base_rate": float(
            labels.mean()
        ),
        "precision@50": precision_at_k(
            labels,
            scores,
            TOP_K,
        ),
        "average_precision": float(
            average_precision_score(
                labels,
                scores,
            )
        ),
        "roc_auc": float(
            roc_auc_score(
                labels,
                scores,
            )
        ),
    }


# ============================================================
# BEFORE — UNSAFE RANDOM-ROW CONTRAST
# ============================================================

all_indices = np.arange(
    len(model_frame)
)

random_train_idx, random_validation_idx = (
    train_test_split(
        all_indices,
        test_size=TEST_SIZE,
        random_state=SEED,
        stratify=model_frame[target_column],
    )
)

random_train = (
    model_frame
    .iloc[random_train_idx]
    .copy()
    .reset_index(drop=True)
)

random_validation = (
    model_frame
    .iloc[random_validation_idx]
    .copy()
    .reset_index(drop=True)
)

random_train_clients = set(
    random_train[group_column].unique()
)

random_validation_clients = set(
    random_validation[group_column].unique()
)

random_client_overlap = len(
    random_train_clients.intersection(
        random_validation_clients
    )
)

random_model = build_audit_model()

random_model.fit(
    random_train[feature_columns],
    random_train[target_column],
)

random_scores = random_model.predict_proba(
    random_validation[feature_columns]
)[:, 1]

random_result = evaluate_split(
    split_name="Before — random row split",
    validation_data=random_validation,
    scores=random_scores,
    training_rows=len(random_train),
    training_clients=len(random_train_clients),
    validation_clients=len(random_validation_clients),
    client_overlap=random_client_overlap,
)


# ============================================================
# AFTER — RETAINED GROUPED-CLIENT DESIGN
# ============================================================

group_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=TEST_SIZE,
    random_state=SEED,
)

group_train_idx, group_validation_idx = next(
    group_splitter.split(
        model_frame[feature_columns],
        model_frame[target_column],
        groups=model_frame[group_column],
    )
)

group_train = (
    model_frame
    .iloc[group_train_idx]
    .copy()
    .reset_index(drop=True)
)

group_validation = (
    model_frame
    .iloc[group_validation_idx]
    .copy()
    .reset_index(drop=True)
)

group_train_clients = set(
    group_train[group_column].unique()
)

group_validation_clients = set(
    group_validation[group_column].unique()
)

group_client_overlap = len(
    group_train_clients.intersection(
        group_validation_clients
    )
)

assert group_client_overlap == 0

grouped_model = build_audit_model()

grouped_model.fit(
    group_train[feature_columns],
    group_train[target_column],
)

grouped_scores = grouped_model.predict_proba(
    group_validation[feature_columns]
)[:, 1]

grouped_result = evaluate_split(
    split_name="After — grouped client split",
    validation_data=group_validation,
    scores=grouped_scores,
    training_rows=len(group_train),
    training_clients=len(group_train_clients),
    validation_clients=len(group_validation_clients),
    client_overlap=group_client_overlap,
)


# ============================================================
# BEFORE / AFTER TABLE
# ============================================================

split_comparison = pd.DataFrame(
    [
        random_result,
        grouped_result,
    ]
)

display(
    split_comparison.round(4)
)

split_comparison.to_csv(
    OUTPUT_DIR
    / "before_after_split_comparison.csv",
    index=False,
)

random_precision_50 = float(
    random_result["precision@50"]
)

grouped_precision_50 = float(
    grouped_result["precision@50"]
)

precision_gap = (
    grouped_precision_50
    - random_precision_50
)

print(
    "Random split Precision@50:",
    round(random_precision_50, 3),
)

print(
    "Grouped split Precision@50:",
    round(grouped_precision_50, 3),
)

print(
    "Grouped minus random difference:",
    round(precision_gap, 3),
)

print(
    "Random split client overlap:",
    random_client_overlap,
)

print(
    "Grouped split client overlap:",
    group_client_overlap,
)

if precision_gap < 0:
    print(
        "Observed result: the grouped estimate is lower. "
        "This is directionally consistent with the random "
        "split being more optimistic."
    )

elif np.isclose(
    precision_gap,
    0,
):
    print(
        "Observed result: the estimates are similar. "
        "The grouped design remains safer because client "
        "overlap is zero."
    )

else:
    print(
        "Observed result: the grouped estimate is higher "
        "on this holdout. The grouped design remains the "
        "honest design because client overlap is zero."
    )

,validation_design,train_rows,validation_rows,train_clients,validation_clients,client_overlap,positive_base_rate,precision@50,average_precision,roc_auc
0,Before — random row split,25528,8510,32,24,24,0.3089,0.50,0.4220,0.6437
1,After — grouped client split,18075,15963,24,8,0,0.3008,0.64,0.3939,0.6175


Random split Precision@50: 0.5
Grouped split Precision@50: 0.64
Grouped minus random difference: 0.14
Random split client overlap: 24
Grouped split client overlap: 0
Observed result: the grouped estimate is higher on this holdout. The grouped design remains the honest design because client overlap is zero.


In [6]:
# ============================================================
# FROZEN ML-07 BASELINE ON GROUPED VALIDATION ROWS
# ============================================================

def percentile_from_training(
    training_values: Any,
    new_values: Any,
) -> np.ndarray:
    training_array = np.asarray(
        training_values,
        dtype=float,
    )

    new_array = np.asarray(
        new_values,
        dtype=float,
    )

    training_array = training_array[
        np.isfinite(training_array)
    ]

    if len(training_array) == 0:
        raise ValueError(
            "Training reference distribution is empty."
        )

    training_array = np.sort(
        training_array
    )

    return (
        np.searchsorted(
            training_array,
            new_array,
            side="right",
        )
        / len(training_array)
    )


baseline_train = group_train.copy()
baseline_validation = group_validation.copy()

training_ctr_reference = (
    baseline_train
    .groupby(
        "position_band",
        observed=True,
    )["feature_ctr"]
    .median()
)

global_training_ctr = float(
    baseline_train["feature_ctr"].median()
)

expected_ctr_train = (
    baseline_train["position_band"]
    .map(training_ctr_reference)
    .astype(float)
    .fillna(global_training_ctr)
)

expected_ctr_validation = (
    baseline_validation["position_band"]
    .map(training_ctr_reference)
    .astype(float)
    .fillna(global_training_ctr)
)

training_ctr_weakness = np.where(
    expected_ctr_train.to_numpy(dtype=float) > 0,
    (
        1.0
        -
        baseline_train["feature_ctr"].to_numpy(dtype=float)
        / expected_ctr_train.to_numpy(dtype=float)
    ),
    0.0,
)

validation_ctr_weakness = np.where(
    expected_ctr_validation.to_numpy(dtype=float) > 0,
    (
        1.0
        -
        baseline_validation["feature_ctr"].to_numpy(dtype=float)
        / expected_ctr_validation.to_numpy(dtype=float)
    ),
    0.0,
)

training_ctr_weakness = np.clip(
    np.nan_to_num(
        training_ctr_weakness,
        nan=0.0,
        posinf=1.0,
        neginf=0.0,
    ),
    0.0,
    1.0,
)

validation_ctr_weakness = np.clip(
    np.nan_to_num(
        validation_ctr_weakness,
        nan=0.0,
        posinf=1.0,
        neginf=0.0,
    ),
    0.0,
    1.0,
)

visibility_percentile = percentile_from_training(
    baseline_train["feature_impressions"],
    baseline_validation["feature_impressions"],
)

baseline_scores = (
    100.0
    * (
        BASELINE_CTR_WEIGHT
        * validation_ctr_weakness
        +
        BASELINE_VISIBILITY_WEIGHT
        * visibility_percentile
    )
)

baseline_scores = np.where(
    validation_ctr_weakness > 0,
    baseline_scores,
    0.0,
)

baseline_scores = np.nan_to_num(
    np.asarray(
        baseline_scores,
        dtype=float,
    ),
    nan=0.0,
    posinf=100.0,
    neginf=0.0,
)

grouped_labels = group_validation[
    target_column
].to_numpy(dtype=int)

baseline_precision_50 = precision_at_k(
    grouped_labels,
    baseline_scores,
    TOP_K,
)

grouped_model_vs_baseline = pd.DataFrame(
    [
        {
            "method": "ML-07 Baseline",
            "validation_rows": len(group_validation),
            "base_rate": float(
                grouped_labels.mean()
            ),
            "precision@50": (
                baseline_precision_50
            ),
            "average_precision": float(
                average_precision_score(
                    grouped_labels,
                    baseline_scores,
                )
            ),
            "roc_auc": float(
                roc_auc_score(
                    grouped_labels,
                    baseline_scores,
                )
            ),
        },
        {
            "method": "Grouped Logistic Regression",
            "validation_rows": len(group_validation),
            "base_rate": float(
                grouped_labels.mean()
            ),
            "precision@50": (
                grouped_precision_50
            ),
            "average_precision": float(
                grouped_result[
                    "average_precision"
                ]
            ),
            "roc_auc": float(
                grouped_result[
                    "roc_auc"
                ]
            ),
        },
    ]
)

display(
    grouped_model_vs_baseline.round(4)
)

grouped_model_vs_baseline.to_csv(
    OUTPUT_DIR
    / "grouped_model_vs_baseline.csv",
    index=False,
)

model_minus_baseline = (
    grouped_precision_50
    - baseline_precision_50
)

print(
    "Grouped model minus baseline Precision@50:",
    round(model_minus_baseline, 3),
)

joblib.dump(
    grouped_model,
    MODEL_PATH,
)

print("Saved grouped model:", MODEL_PATH)

,method,validation_rows,base_rate,precision@50,average_precision,roc_auc
0,ML-07 Baseline,15963,0.3008,0.52,0.3776,0.6059
1,Grouped Logistic Regression,15963,0.3008,0.64,0.3939,0.6175


Grouped model minus baseline Precision@50: 0.12
Saved grouped model: work/outputs/ml09/models/ml09_grouped_logistic_regression.joblib


## 3. Leakage Audit

The final Week-5 feature set is audited against four leakage classes:

1. **Label-derived fields** — fields used to create or directly reveal the target;
2. **Future or overlapping windows** — fields measured during the outcome period;
3. **Decision-derived product fields** — existing scores, flags, priorities, or actions;
4. **Identifiers and private data** — client/content IDs are grouping or joining keys only.

The feature timeline is:

- feature measurements: March 1–15, 2026;
- decision moment: after March 15, 2026;
- outcome proxy: March 16–31, 2026.

The outcome availability field is used only to define a retrospectively measurable evaluation set. It is not a model feature.

To verify that the audit harness can detect leakage, this section deliberately adds `decline_ratio`, a field constructed from the outcome period. Its performance is compared with the honest grouped model, and the field is then excluded from the retained feature set.

Real grouped-holdout failure examples are also inspected without displaying pseudonymous IDs.

In [7]:
# ============================================================
# FEATURE INVENTORY
# ============================================================

feature_inventory = pd.DataFrame(
    [
        {
            "feature": "log_feature_impressions",
            "source_window": "March 1–15",
            "available_at_decision": True,
            "role": "Model feature",
            "leakage_risk": "None observed",
        },
        {
            "feature": "feature_clicks",
            "source_window": "March 1–15",
            "available_at_decision": True,
            "role": "Model feature",
            "leakage_risk": "None observed",
        },
        {
            "feature": "feature_ctr",
            "source_window": "March 1–15",
            "available_at_decision": True,
            "role": "Model feature",
            "leakage_risk": "None observed",
        },
        {
            "feature": "feature_avg_position",
            "source_window": "March 1–15",
            "available_at_decision": True,
            "role": "Model feature",
            "leakage_risk": "None observed",
        },
        {
            "feature": "feature_active_days",
            "source_window": "March 1–15",
            "available_at_decision": True,
            "role": "Model feature",
            "leakage_risk": "None observed",
        },
        {
            "feature": "feature_position_volatility",
            "source_window": "March 1–15",
            "available_at_decision": True,
            "role": "Model feature",
            "leakage_risk": (
                "Training-only median imputation required"
            ),
        },
        {
            "feature": "position_band",
            "source_window": "March 1–15",
            "available_at_decision": True,
            "role": "Model feature",
            "leakage_risk": (
                "Derived from safe pre-outcome position"
            ),
        },
    ]
)

display(feature_inventory)


# ============================================================
# STATIC LEAKAGE CHECKS
# ============================================================

label_or_future_fields = {
    "outcome_impressions",
    "outcome_daily_impressions",
    "outcome_available_days",
    "is_declining_proxy",
    "decline_ratio",
    "trend_direction",
    "trend_pct",
}

decision_fields = {
    "health_score",
    "priority_score",
    "action_type",
    "needs_ctr_fix",
    "refresh_flag",
    "baseline_action_score",
}

identifier_fields = {
    "client_hash_id",
    "content_hash_id",
}

privacy_patterns = [
    "url",
    "domain",
    "query_text",
    "raw_query",
    "client_name",
    "email",
    "token",
]

feature_set = set(feature_columns)

label_or_future_overlap = sorted(
    feature_set.intersection(
        label_or_future_fields
    )
)

decision_overlap = sorted(
    feature_set.intersection(
        decision_fields
    )
)

identifier_overlap = sorted(
    feature_set.intersection(
        identifier_fields
    )
)

privacy_overlap = sorted(
    feature
    for feature in feature_columns
    if any(
        pattern in feature.lower()
        for pattern in privacy_patterns
    )
)

leakage_audit = pd.DataFrame(
    [
        {
            "check": (
                "No label-derived or future fields"
            ),
            "matches": label_or_future_overlap,
            "passed": (
                len(label_or_future_overlap) == 0
            ),
        },
        {
            "check": (
                "No decision-derived product fields"
            ),
            "matches": decision_overlap,
            "passed": (
                len(decision_overlap) == 0
            ),
        },
        {
            "check": (
                "No identifiers in model features"
            ),
            "matches": identifier_overlap,
            "passed": (
                len(identifier_overlap) == 0
            ),
        },
        {
            "check": (
                "No privacy-sensitive feature names"
            ),
            "matches": privacy_overlap,
            "passed": (
                len(privacy_overlap) == 0
            ),
        },
        {
            "check": (
                "All features are available before outcome"
            ),
            "matches": [],
            "passed": bool(
                feature_inventory[
                    "available_at_decision"
                ].all()
            ),
        },
    ]
)

display(leakage_audit)

assert leakage_audit["passed"].all()

leakage_audit.to_csv(
    OUTPUT_DIR
    / "feature_leakage_audit.csv",
    index=False,
)

,feature,source_window,available_at_decision,role,leakage_risk
0,log_feature_impressions,March 1–15,True,Model feature,None observed
1,feature_clicks,March 1–15,True,Model feature,None observed
2,feature_ctr,March 1–15,True,Model feature,None observed
3,feature_avg_position,March 1–15,True,Model feature,None observed
4,feature_active_days,March 1–15,True,Model feature,None observed
5,feature_position_volatility,March 1–15,True,Model feature,Training-only median imputation required
6,position_band,March 1–15,True,Model feature,Derived from safe pre-outcome position


,check,matches,passed
0,No label-derived or future fields,[],True
1,No decision-derived product fields,[],True
2,No identifiers in model features,[],True
3,No privacy-sensitive feature names,[],True
4,All features are available before outcome,[],True


In [8]:
# ============================================================
# DELIBERATE LEAK TEST
# ============================================================

leaky_frame = model_frame.copy()

leaky_frame["decline_ratio"] = (
    leaky_frame["outcome_daily_impressions"]
    /
    leaky_frame["feature_daily_impressions"]
)

leaky_numeric_features = [
    *numeric_features,
    "decline_ratio",
]

leaky_feature_columns = [
    *leaky_numeric_features,
    *categorical_features,
]

leaky_numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            ),
        ),
        (
            "scaler",
            StandardScaler(),
        ),
    ]
)

leaky_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            leaky_numeric_pipeline,
            leaky_numeric_features,
        ),
        (
            "categorical",
            clone(categorical_pipeline),
            categorical_features,
        ),
    ],
    remainder="drop",
)

leaky_model = Pipeline(
    steps=[
        (
            "preprocessor",
            leaky_preprocessor,
        ),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=SEED,
            ),
        ),
    ]
)

leaky_train = (
    leaky_frame
    .iloc[group_train_idx]
    .copy()
    .reset_index(drop=True)
)

leaky_validation = (
    leaky_frame
    .iloc[group_validation_idx]
    .copy()
    .reset_index(drop=True)
)

leaky_model.fit(
    leaky_train[leaky_feature_columns],
    leaky_train[target_column],
)

leaky_scores = leaky_model.predict_proba(
    leaky_validation[leaky_feature_columns]
)[:, 1]

honest_vs_leaky = pd.DataFrame(
    [
        {
            "feature_set": (
                "Honest pre-outcome features"
            ),
            "precision@50": (
                grouped_precision_50
            ),
            "average_precision": float(
                grouped_result[
                    "average_precision"
                ]
            ),
            "roc_auc": float(
                grouped_result[
                    "roc_auc"
                ]
            ),
            "contains_outcome_field": False,
        },
        {
            "feature_set": (
                "Leaky features plus decline_ratio"
            ),
            "precision@50": precision_at_k(
                grouped_labels,
                leaky_scores,
                TOP_K,
            ),
            "average_precision": float(
                average_precision_score(
                    grouped_labels,
                    leaky_scores,
                )
            ),
            "roc_auc": float(
                roc_auc_score(
                    grouped_labels,
                    leaky_scores,
                )
            ),
            "contains_outcome_field": True,
        },
    ]
)

display(
    honest_vs_leaky.round(4)
)

honest_vs_leaky.to_csv(
    OUTPUT_DIR
    / "honest_vs_leaky.csv",
    index=False,
)

leaky_average_precision = float(
    honest_vs_leaky.loc[
        honest_vs_leaky[
            "contains_outcome_field"
        ],
        "average_precision",
    ].iloc[0]
)

honest_average_precision = float(
    honest_vs_leaky.loc[
        ~honest_vs_leaky[
            "contains_outcome_field"
        ],
        "average_precision",
    ].iloc[0]
)

assert "decline_ratio" not in feature_columns
assert leaky_average_precision >= honest_average_precision

print(
    "Deliberate leak detected. "
    "decline_ratio is excluded from the retained model."
)

,feature_set,precision@50,average_precision,roc_auc,contains_outcome_field
0,Honest pre-outcome features,0.64,0.3939,0.6175,False
1,Leaky features plus decline_ratio,1.00,0.9999,1.0000,True


Deliberate leak detected. decline_ratio is excluded from the retained model.


In [9]:
# ============================================================
# REAL GROUPED-HOLDOUT FAILURE EXAMPLES
# ============================================================

grouped_predictions = (
    np.asarray(
        grouped_scores,
        dtype=float,
    )
    >=
    CLASSIFICATION_THRESHOLD
).astype(int)

grouped_confusion = confusion_matrix(
    grouped_labels,
    grouped_predictions,
    labels=[
        0,
        1,
    ],
)

grouped_confusion_table = pd.DataFrame(
    grouped_confusion,
    index=[
        "Actual not declining",
        "Actual declining",
    ],
    columns=[
        "Predicted not declining",
        "Predicted declining",
    ],
)

display(grouped_confusion_table)

error_frame = group_validation[
    [
        "feature_impressions",
        "feature_clicks",
        "feature_ctr",
        "feature_avg_position",
        "feature_active_days",
        "feature_position_volatility",
        "position_band",
        target_column,
    ]
].copy()

error_frame["model_score"] = grouped_scores
error_frame["predicted_at_0_5"] = grouped_predictions

error_frame["error_type"] = np.select(
    [
        (
            error_frame[target_column].eq(0)
            &
            error_frame[
                "predicted_at_0_5"
            ].eq(1)
        ),
        (
            error_frame[target_column].eq(1)
            &
            error_frame[
                "predicted_at_0_5"
            ].eq(0)
        ),
    ],
    [
        "false_positive",
        "false_negative",
    ],
    default="correct",
)

false_positives = (
    error_frame[
        error_frame["error_type"].eq(
            "false_positive"
        )
    ]
    .sort_values(
        "model_score",
        ascending=False,
    )
    .reset_index(drop=True)
)

false_negatives = (
    error_frame[
        error_frame["error_type"].eq(
            "false_negative"
        )
    ]
    .sort_values(
        "model_score",
        ascending=False,
    )
    .reset_index(drop=True)
)

failure_examples = pd.concat(
    [
        false_positives.head(5),
        false_negatives.head(5),
    ],
    ignore_index=True,
)

public_failure_columns = [
    "error_type",
    "model_score",
    target_column,
    "feature_impressions",
    "feature_ctr",
    "feature_avg_position",
    "feature_active_days",
    "feature_position_volatility",
    "position_band",
]

display(
    failure_examples[
        public_failure_columns
    ].round(5)
)

failure_examples[
    public_failure_columns
].to_csv(
    OUTPUT_DIR
    / "failure_examples.csv",
    index=False,
)

false_positive_count = int(
    len(false_positives)
)

false_negative_count = int(
    len(false_negatives)
)

print(
    "Grouped-holdout false positives:",
    f"{false_positive_count:,}",
)

print(
    "Grouped-holdout false negatives:",
    f"{false_negative_count:,}",
)

print(
    "Interpretation: some high-scoring pages did not meet "
    "the later decline proxy, while some declining-proxy "
    "pages remained near or below the 0.50 diagnostic "
    "threshold. This limits automatic use."
)

,Predicted not declining,Predicted declining
Actual not declining,5880,5282
Actual declining,1705,3096


,error_type,model_score,is_declining_proxy,feature_impressions,feature_ctr,feature_avg_position,feature_active_days,feature_position_volatility,position_band
0,false_positive,0.74321,0,943.0,0.00000,8.38256,15,11.92102,Page 1
1,false_positive,0.73774,0,740.0,0.00000,8.89489,15,10.64815,Page 1
2,false_positive,0.73251,0,676.0,0.00000,8.70931,15,10.07596,Page 1
3,false_positive,0.72647,0,544.0,0.00000,9.58261,15,8.53360,Page 1
4,false_positive,0.72601,0,1008.0,0.00000,9.54791,14,9.84497,Page 1
5,false_negative,0.49994,1,1012.0,0.00296,1.12066,15,0.35476,Top 3
6,false_negative,0.49989,1,534.0,0.00375,1.81975,15,1.21458,Top 3
7,false_negative,0.49989,1,23705.0,0.00194,4.74658,15,0.41118,Page 1
8,false_negative,0.49987,1,1523.0,0.00328,4.26875,15,0.73878,Page 1
9,false_negative,0.49987,1,506.0,0.00593,6.49407,15,5.03426,Page 1


Grouped-holdout false positives: 5,282
Grouped-holdout false negatives: 1,705
Interpretation: some high-scoring pages did not meet the later decline proxy, while some declining-proxy pages remained near or below the 0.50 diagnostic threshold. This limits automatic use.


## 4. Claim Rewrite

### Bold claim that goes beyond the evidence

> The model reliably predicts which pages will decline and should be refreshed.

This sentence overstates both prediction certainty and the action implication. The target is a retrospective proxy, the validation uses one grouped holdout, and no refresh intervention was tested.

### Public-safe rewrite

The executed code below inserts the measured grouped-client model and baseline results into a careful claim.

The rewritten statement:

- reports the validation design;
- reports Precision@50 beside the baseline;
- uses measured and observed language;
- frames the result as directional decision-support;
- excludes causal, guaranteed, and automatic-action claims.

In [10]:
safe_claim = (
    "On one grouped-client holdout, Logistic Regression "
    f"measured a Precision@50 of "
    f"{grouped_precision_50:.3f}, compared with "
    f"{baseline_precision_50:.3f} for the frozen ML-07 "
    f"baseline, an absolute measured difference of "
    f"{model_minus_baseline:.3f}. "
    "This observed result supports directional human "
    "decision-support for content-review prioritisation "
    "within the March 2026 evaluation design. It does not "
    "establish causal refresh impact, guarantee performance "
    "for future clients or time periods, or support "
    "automatic content changes."
)

claim_rewrite = pd.DataFrame(
    [
        {
            "claim_type": "Overclaim",
            "claim": (
                "The model reliably predicts which pages "
                "will decline and should be refreshed."
            ),
        },
        {
            "claim_type": "Public-safe rewrite",
            "claim": safe_claim,
        },
    ]
)

display(claim_rewrite)

print("PUBLIC-SAFE CLAIM")
print("-" * 80)
print(safe_claim)

required_claim_terms = [
    "measured",
    "observed",
    "directional",
    "decision-support",
]

assert all(
    term.lower() in safe_claim.lower()
    for term in required_claim_terms
)

,claim_type,claim
0,Overclaim,The model reliably predicts which pages will d...
1,Public-safe rewrite,"On one grouped-client holdout, Logistic Regres..."


PUBLIC-SAFE CLAIM
--------------------------------------------------------------------------------
On one grouped-client holdout, Logistic Regression measured a Precision@50 of 0.640, compared with 0.520 for the frozen ML-07 baseline, an absolute measured difference of 0.120. This observed result supports directional human decision-support for content-review prioritisation within the March 2026 evaluation design. It does not establish causal refresh impact, guarantee performance for future clients or time periods, or support automatic content changes.


In [11]:
# ============================================================
# AGGREGATE METRICS RECEIPT
# ============================================================

metrics_receipt = {
    "assignment": (
        "ML-09 Validation and Research Claim Audit"
    ),
    "lane": (
        "Refresh / Content Opportunity Scoring"
    ),
    "development_month": (
        "March 2026"
    ),
    "feature_window": (
        f"{FEATURE_START} to {FEATURE_END}"
    ),
    "outcome_window": (
        f"{OUTCOME_START} to {OUTCOME_END}"
    ),
    "operational_rows": int(
        len(model_frame)
    ),
    "anonymised_clients": int(
        model_frame[group_column].nunique()
    ),
    "random_split": random_result,
    "grouped_split": grouped_result,
    "random_minus_grouped_precision_at_50": float(
        random_precision_50
        - grouped_precision_50
    ),
    "grouped_baseline_precision_at_50": float(
        baseline_precision_50
    ),
    "grouped_model_minus_baseline_precision_at_50": float(
        model_minus_baseline
    ),
    "leaky_average_precision": float(
        leaky_average_precision
    ),
    "honest_average_precision": float(
        honest_average_precision
    ),
    "false_positive_count_at_0_5": (
        false_positive_count
    ),
    "false_negative_count_at_0_5": (
        false_negative_count
    ),
    "paper_findings_reviewed": 2,
    "feature_leakage_checks_passed": bool(
        leakage_audit["passed"].all()
    ),
    "retained_validation_design": (
        "Grouped client holdout"
    ),
    "safe_claim": safe_claim,
    "automatic_action": False,
    "causal_claim": False,
}

with METRICS_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metrics_receipt,
        file,
        indent=2,
    )

print("Saved metrics receipt:", METRICS_PATH)

Saved metrics receipt: work/outputs/ml09/ml09_metrics.json


## Self-check

Before submission, this notebook confirms that:

- two paper findings and constructive methodology questions are documented;
- the notebook is standalone and does not execute another notebook;
- the Week-5 model is reconstructed on the same operational population;
- random-row and grouped-client estimates are reported together;
- the positive base rate is printed beside the metrics;
- grouped-client overlap is zero;
- training-only preprocessing is used;
- label-derived, future, decision-derived, identifier, and privacy leakage checks pass;
- a deliberate leaky field causes a measurable score increase and is then excluded;
- real false-positive and false-negative examples are displayed without IDs;
- the strongest claim is rewritten using observed, measured, directional, and decision-support language;
- no causal or automatic-action claim is made;
- aggregate receipts and the grouped model are exported;
- the notebook runs from top to bottom without errors.

In [12]:
required_outputs = [
    OUTPUT_DIR
    / "before_after_split_comparison.csv",

    OUTPUT_DIR
    / "grouped_model_vs_baseline.csv",

    OUTPUT_DIR
    / "feature_leakage_audit.csv",

    OUTPUT_DIR
    / "honest_vs_leaky.csv",

    OUTPUT_DIR
    / "failure_examples.csv",

    METRICS_PATH,

    MODEL_PATH,
]

public_failure_set = set(
    public_failure_columns
)

self_checks = {
    "Two paper findings documented": (
        len(paper_findings) == 2
    ),

    "Paper interpretations are observational": (
        paper_findings[
            "evidence_type"
        ].str.contains(
            "observational",
            case=False,
        ).all()
    ),

    "Modeling frame is non-empty": (
        len(model_frame) > 0
    ),

    "Seven final features are used": (
        len(feature_columns) == 7
    ),

    "Random split has client overlap": (
        random_client_overlap > 0
    ),

    "Grouped split has zero client overlap": (
        group_client_overlap == 0
    ),

    "Both split results include base rate": (
        split_comparison[
            "positive_base_rate"
        ].between(
            0,
            1,
        ).all()
    ),

    "Both split scores are finite": (
        np.isfinite(
            split_comparison[
                [
                    "precision@50",
                    "average_precision",
                    "roc_auc",
                ]
            ].to_numpy(
                dtype=float
            )
        ).all()
    ),

    "Frozen baseline uses grouped validation rows": (
        len(baseline_scores)
        ==
        len(group_validation)
    ),

    "Feature leakage audit passes": (
        leakage_audit["passed"].all()
    ),

    "No outcome field in honest features": (
        not bool(
            set(feature_columns).intersection(
                label_or_future_fields
            )
        )
    ),

    "No identifier in honest features": (
        not bool(
            set(feature_columns).intersection(
                identifier_fields
            )
        )
    ),

    "Deliberate leaky field is excluded": (
        "decline_ratio"
        not in feature_columns
    ),

    "Leaky AP is not lower than honest AP": (
        leaky_average_precision
        >=
        honest_average_precision
    ),

    "At least three false positives exist": (
        false_positive_count >= 3
    ),

    "At least three false negatives exist": (
        false_negative_count >= 3
    ),

    "Failure display excludes identifiers": (
        not bool(
            public_failure_set.intersection(
                identifier_fields
            )
        )
    ),

    "Safe claim contains required language": all(
        term.lower() in safe_claim.lower()
        for term in required_claim_terms
    ),

    "No automatic action": (
        metrics_receipt[
            "automatic_action"
        ]
        is False
    ),

    "No causal claim": (
        metrics_receipt[
            "causal_claim"
        ]
        is False
    ),

    "Required outputs exist": all(
        path.exists()
        for path in required_outputs
    ),
}

self_check_df = pd.DataFrame(
    {
        "check": list(
            self_checks.keys()
        ),
        "passed": list(
            self_checks.values()
        ),
    }
)

display(self_check_df)

failed_checks = [
    check_name
    for check_name, passed
    in self_checks.items()
    if not passed
]

if failed_checks:
    raise AssertionError(
        "ML-09 self-check failed: "
        + ", ".join(failed_checks)
    )

print("All ML-09 self-checks passed.")

,check,passed
0,Two paper findings documented,True
1,Paper interpretations are observational,True
2,Modeling frame is non-empty,True
3,Seven final features are used,True
4,Random split has client overlap,True
5,Grouped split has zero client overlap,True
6,Both split results include base rate,True
7,Both split scores are finite,True
8,Frozen baseline uses grouped validation rows,True
9,Feature leakage audit passes,True


All ML-09 self-checks passed.
